# 1-3. 임베딩 심화

⏱ **소요시간**: 30분

## 학습 목표
- Static → Contextual → Sentence 임베딩의 진화를 설명할 수 있다.
- cosine / dot / L2 유사도의 수식과 정규화의 의미를 알 수 있다.
- 한국어 RAG에서 자주 쓰이는 3개 모델(bge-m3, ko-sroberta, multilingual-e5)를 동일 입력으로 비교한다.
- MTEB-ko 리더보드의 구조를 이해한다.

## 핵심 키워드
`word2vec`, `BERT CLS`, `SBERT`, `SimCSE`, `MTEB-ko`, `cosine/dot/L2`

## 1. 임베딩의 진화

### 1세대 — Static Word Embedding (2013~)
- **word2vec** (Mikolov 2013), **GloVe**, **FastText**
- 단어 1개 = 고정된 벡터 1개. 문맥에 따라 바뀌지 않는다.
- **한계**: "배(ship)" vs "배(pear)" vs "배(stomach)" 동형이의어를 구분하지 못함.

### 2세대 — Contextual Embedding (2018~)
- **ELMo**, **BERT** — 같은 단어도 문장에 따라 다른 벡터.
- BERT의 `[CLS]` 토큰이 문장 대표 임베딩으로 흔히 사용됨.
- **한계**: 원래 BERT의 `[CLS]`는 유사도용으로 학습된 게 아니라 직접 cosine에 넣으면 성능이 낮다.

### 3세대 — Sentence Embedding (2019~)
- **SBERT** (Sentence-BERT, 2019) — 문장 쌍 유사도로 fine-tune. Siamese/triplet loss.
- **SimCSE** (2021) — dropout 다른 것끼리 positive pair로 contrastive 학습.
- **bge / e5 / ko-sroberta** (2023→) — 대규모 다중 언어 데이터에 contrastive + hard negatives.
- 최근 트렌드: **query · passage instruction prefix** (“query: …” / “passage: …”)

## 2. 유사도 수식

### cosine similarity
$$
\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \lVert \mathbf{b} \rVert} \in [-1, 1]
$$

### dot product
$$
\text{dot}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b} = \sum_i a_i b_i
$$

### Euclidean (L2)
$$
d_2(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}
$$

### 정규화가 중요한 이유
모든 벡터를 $\lVert \mathbf{v} \rVert = 1$ 로 정규화하면:
- $\cos(a, b) = a \cdot b$ (dot과 같아짐 → FAISS 같은 벡터스토어는 inner product만 지원해도 cosine 효과)
- $d_2(a, b)^2 = 2 - 2\cos(a, b)$ → cosine 순서와 L2 순서 동일

따라서 **얻은 임베딩을 반드시 L2-normalize 하고 저장하는 것**이 실무 표준이다.

In [ ]:
import sys
sys.path.insert(0, '../..')
import numpy as np
from common import provider_badge

print(provider_badge())

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def dot(a, b):
    return np.dot(a, b)


def l2(a, b):
    return np.linalg.norm(a - b)


# 빠른 확인: 단위벡터에서는 cosine 순서 == -L2 순서
rng = np.random.default_rng(0)
vs = rng.standard_normal((5, 16))
vs_norm = vs / np.linalg.norm(vs, axis=1, keepdims=True)
q = vs_norm[0]

print(f'{"idx":>4} {"cos":>8} {"dot":>8} {"L2":>8}')
for i, v in enumerate(vs_norm):
    print(f'{i:>4} {cosine(q, v):>8.3f} {dot(q, v):>8.3f} {l2(q, v):>8.3f}')

## 3. 한국어 문장 임베딩 모델 3종 비교

| 모델 | 차원 | 특징 |
|---|---|---|
| `BAAI/bge-m3` | 1024 | 다중 언어, dense+sparse+colbert 삼중 출력 |
| `jhgan/ko-sroberta-multitask` | 768 | 한국어 전용 SBERT, 한국어 NLI/STS 학습 |
| `intfloat/multilingual-e5-large-instruct` | 1024 | query/passage instruction prefix 필요 |

In [ ]:
from sentence_transformers import SentenceTransformer

MODELS_TO_COMPARE = {
    'bge-m3': 'BAAI/bge-m3',
    'ko-sroberta': 'jhgan/ko-sroberta-multitask',
    'multilingual-e5': 'intfloat/multilingual-e5-large-instruct',
}

# 처음 로드 시 다운로드로 시간이 조금 걸림. 폐쇄망 에서는 사전 캐시된 폴더 사용.
models = {}
for name, repo in MODELS_TO_COMPARE.items():
    print(f'loading {name}...')
    models[name] = SentenceTransformer(repo)
print('done.')

In [ ]:
# 테스트 문장쌍 — 금융/규제 도메인
queries = [
    '망분리 규제에서 외부 클라우드 사용은 가능한가?',
    '전자금융거래에서 소비자 보호는 어떻게 이루어지나?',
    'LLM 환각 현상을 줄이는 방법은 무엇인가?',
]

candidates = [
    # 0: 망분리 규제 관련
    '전자금융감독규정은 전산실과 정보처리시스템을 인터넷과 분리된 망에서 운영하도록 규정한다.',
    # 1: 소비자 보호 관련
    '전자금융거래법은 이용자 권익 보호, 사고 시 피해 보상과 이의 제기 절차를 규정한다.',
    # 2: 환각 완화
    'RAG는 외부 지식베이스에서 관련 문서를 검색한 뒤 그 내용을 컨텍스트로 넣어 환각을 줄이는 기법이다.',
    # 분산 소재들 (부정답)
    '서울의 날씨는 오늘 맑고 낮 기온은 20도입니다.',
    '단위 행렬을 직교화 분해하는 선형대수 수업을 수강하세요.',
]

In [ ]:
def embed_with_instruction(model_name, model, texts, role='passage'):
    """e5 계열은 instruction prefix가 필요하다.

    role: 'query' 또는 'passage'
    """
    if model_name == 'multilingual-e5':
        prefixed = [f'{role}: {t}' for t in texts]
        return model.encode(prefixed, normalize_embeddings=True)
    return model.encode(texts, normalize_embeddings=True)


print(f'{"model":<18} {"query_idx":<10} ' + ' '.join([f'c{i}' for i in range(len(candidates))]))
print('-' * 80)
for name, model in models.items():
    q_emb = embed_with_instruction(name, model, queries, role='query')
    c_emb = embed_with_instruction(name, model, candidates, role='passage')
    sims = q_emb @ c_emb.T  # 정규화 되었으므로 cosine == dot
    for qi in range(len(queries)):
        row = ' '.join(f'{sims[qi, ci]:>5.2f}' for ci in range(len(candidates)))
        print(f'{name:<18} q{qi:<9} {row}')
    print()

기대 패턴: 각 질의(`q0`, `q1`, `q2`)에 대해 동일 인덱스의 콘텐츠(`c0`, `c1`, `c2`)가 가장 높은 유사도를 가져야 한다. 마지막 두 소재(날씨, 수학)는 모두 낮아야 함.

## 4. 비교 메트릭: Retrieval Accuracy @ 1

각 질의가 실제 정답(인덱스 0·1·2의 소재)을 top-1으로 맞춰내는지 확인한다.

In [ ]:
gold = [0, 1, 2]  # queries[i] 의 정답은 candidates[gold[i]]

print(f'{"model":<18} hit@1  top1_sims')
for name, model in models.items():
    q_emb = embed_with_instruction(name, model, queries, role='query')
    c_emb = embed_with_instruction(name, model, candidates, role='passage')
    sims = q_emb @ c_emb.T
    top1 = sims.argmax(axis=1)
    hits = (top1 == np.array(gold)).sum()
    top1_sims = [f'{sims[i, top1[i]]:.3f}' for i in range(len(queries))]
    print(f'{name:<18} {hits}/{len(queries)}    {top1_sims}  (top1_idx={list(top1)})')

<!-- optional -->

## 5. cosine vs dot 시각화

정규화 한 순간 cosine과 dot이 동일하지만, 정규화하지 않으면 **벡터 크기에 편향**된다.

In [ ]:
# <!-- optional -->
# ko-sroberta로 정규화 전/후 차이를 확인
model = models['ko-sroberta']
raw = model.encode(candidates, normalize_embeddings=False)
norms = np.linalg.norm(raw, axis=1)
print('raw 벡터 norm (문서마다 크기 다름):', norms)

q_raw = model.encode([queries[0]], normalize_embeddings=False)[0]

dot_scores = raw @ q_raw
cos_scores = dot_scores / (norms * np.linalg.norm(q_raw))

print()
print(f'{"idx":>4} {"dot":>10} {"cos":>8}')
for i in range(len(candidates)):
    print(f'{i:>4} {dot_scores[i]:>10.3f} {cos_scores[i]:>8.3f}')
print()
print('=> 정규화 안 하면 긴 문서가 dot-score에서 유리한 것처럼 보일 수 있다.')

## 6. MTEB-ko 리더보드

- **MTEB (Massive Text Embedding Benchmark)** — Muennighoff et al. 2023.
- 다양한 task (Retrieval, STS, Classification, Clustering 등)에 대해 동일 모델을 평가.
- **MTEB-ko**는 한국어 전용 태스크 모음. 리트리버 문서의 링크:
  - HuggingFace MTEB Leaderboard: https://huggingface.co/spaces/mteb/leaderboard
  - 선택: Language = `kor` 으로 필터

현시점 상위 모델 (2024-2025 기준 상위권):
1. `BAAI/bge-m3` — 범용, 여전히 강력
2. `dragonkue/snowflake-arctic-embed-l-v2.0-ko` — 한국어 특화 fine-tune
3. `nlpai-lab/KURE-v1` — 고려대 NLP Lab 릴리스
4. `intfloat/multilingual-e5-large-instruct` — instruction 기반

폐쇄망 권장 선택 관점:
- **리트리버 점수 유사하면** 모델 크기 · 추론 속도 · 클라이언트 GPU에 맞는지가 더 중요.
- **Rerank**와 결합하면 1차 모델 선택의 영향이 다소 완화된다 (Day 2 S3-1).

## 과제

1. 사내 FAQ/계약서 질의 20개와 정답 소재 매핑을 준비해 위 3개 모델의 hit@1, MRR@5를 측정해 도메인에 가장 잘 맞는 모델을 골라보세요.
2. `multilingual-e5` 로서 실수로 **instruction prefix를 빼면** hit@1이 얼마나 떨어지는지 실험해보세요.
3. `normalize_embeddings=False` 로 벡터를 생성해 FAISS inner-product 인덱스를 넣을 경우와, `True`로 생성한 벡터를 넣을 경우 검색 결과가 어떻게 달라지는지 확인하세요.

## 더 읽어보기
- Reimers & Gurevych, [Sentence-BERT (2019)](https://arxiv.org/abs/1908.10084)
- Gao et al., [SimCSE (2021)](https://arxiv.org/abs/2104.08821)
- Chen et al., [BGE M3-Embedding (2024)](https://arxiv.org/abs/2402.03216)
- [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) (한국어 필터에서 `kor` 선택)
- [langchain-kr · 08-Embedding](https://github.com/teddylee777/langchain-kr/tree/main/08-Embedding)